In [1]:
import pandas as pd

df = pd.read_csv('data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv')
df.shape

(1470, 35)

In [2]:
# 1. Drop constant/useless columns
df_clean = df.drop(columns=['EmployeeCount', 'Over18', 'StandardHours'])

# 2. Create binary attrition flag for modeling
df_clean['AttritionFlag'] = df_clean['Attrition'].map({'Yes': 1, 'No': 0})

# 3. Map ordinal-encoded columns to readable labels
education_map = {1: 'Below College', 2: 'College', 3: 'Bachelor', 4: 'Master', 5: 'Doctor'}
satisfaction_map = {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'}
worklife_map = {1: 'Bad', 2: 'Good', 3: 'Better', 4: 'Best'}
performance_map = {1: 'Low', 2: 'Good', 3: 'Excellent', 4: 'Outstanding'}

df_clean['EducationLabel'] = df_clean['Education'].map(education_map)
df_clean['EnvironmentSatisfactionLabel'] = df_clean['EnvironmentSatisfaction'].map(satisfaction_map)
df_clean['JobInvolvementLabel'] = df_clean['JobInvolvement'].map(satisfaction_map)
df_clean['JobSatisfactionLabel'] = df_clean['JobSatisfaction'].map(satisfaction_map)
df_clean['RelationshipSatisfactionLabel'] = df_clean['RelationshipSatisfaction'].map(satisfaction_map)
df_clean['WorkLifeBalanceLabel'] = df_clean['WorkLifeBalance'].map(worklife_map)
df_clean['PerformanceRatingLabel'] = df_clean['PerformanceRating'].map(performance_map)

# 4. Age buckets
age_bins = [17, 25, 35, 45, 55, 61]
age_labels = ['18-25', '26-35', '36-45', '46-55', '56-60']
df_clean['AgeGroup'] = pd.cut(df_clean['Age'], bins=age_bins, labels=age_labels)

# 5. Tenure buckets
tenure_bins = [-1, 2, 5, 10, 20, 100]
tenure_labels = ['0-2 yrs', '3-5 yrs', '6-10 yrs', '11-20 yrs', '20+ yrs']
df_clean['TenureGroup'] = pd.cut(df_clean['YearsAtCompany'], bins=tenure_bins, labels=tenure_labels)

df_clean.shape

(1470, 42)

In [3]:
import os
os.makedirs('data/processed', exist_ok=True)
df_clean.to_csv('data/processed/hr_attrition_clean.csv', index=False)

In [4]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)
db_url = os.getenv('DATABASE_URL')
print(db_url is not None)

True


In [5]:
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine(db_url)
df_clean.to_sql('employees', engine, if_exists='replace', index=False)

check = pd.read_sql('SELECT COUNT(*) FROM employees', engine)
print(check)

   count
0   1470


In [6]:
q1 = '''
SELECT 
    COUNT(*) AS total_employees,
    SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) AS attrition_count,
    ROUND(100.0 * SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS attrition_rate_pct
FROM employees;
'''
pd.read_sql(q1, engine)

,total_employees,attrition_count,attrition_rate_pct
0,1470,237,16.12


In [7]:
q2 = '''
SELECT 
    "Department",
    COUNT(*) AS total,
    SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) AS attrition_count,
    ROUND(100.0 * SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS attrition_rate_pct
FROM employees
GROUP BY "Department"
ORDER BY attrition_rate_pct DESC;
'''
pd.read_sql(q2, engine)

,Department,total,attrition_count,attrition_rate_pct
0,Sales,446,92,20.63
1,Human Resources,63,12,19.05
2,Research & Development,961,133,13.84


In [8]:
q3 = '''
SELECT 
    "TenureGroup",
    COUNT(*) AS total,
    ROUND(100.0 * SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS attrition_rate_pct
FROM employees
GROUP BY "TenureGroup"
ORDER BY "TenureGroup";
'''
pd.read_sql(q3, engine)

,TenureGroup,total,attrition_rate_pct
0,0-2 yrs,342,29.82
1,11-20 yrs,180,6.67
2,20+ yrs,66,12.12
3,3-5 yrs,434,13.82
4,6-10 yrs,448,12.28


In [9]:
q4 = '''
SELECT 
    "JobRole",
    COUNT(*) AS attritions,
    ROUND(AVG("MonthlyIncome"), 0) AS avg_monthly_income_lost
FROM employees
WHERE "Attrition" = 'Yes'
GROUP BY "JobRole"
ORDER BY attritions DESC;
'''
pd.read_sql(q4, engine)

,JobRole,attritions,avg_monthly_income_lost
0,Laboratory Technician,62,2919.0
1,Sales Executive,57,7489.0
2,Research Scientist,47,2780.0
3,Sales Representative,33,2365.0
4,Human Resources,12,3716.0
5,Manufacturing Director,10,7366.0
6,Healthcare Representative,9,8548.0
7,Manager,5,16797.0
8,Research Director,2,19396.0


In [10]:
q5 = '''
SELECT 
    "OverTime",
    COUNT(*) AS total,
    ROUND(100.0 * SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS attrition_rate_pct
FROM employees
GROUP BY "OverTime";
'''
pd.read_sql(q5, engine)

,OverTime,total,attrition_rate_pct
0,No,1054,10.44
1,Yes,416,30.53


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Select features - drop identifiers, target, and label columns we engineered for display only
drop_cols = ['Attrition', 'AttritionFlag', 'EmployeeNumber', 'EducationLabel', 
             'EnvironmentSatisfactionLabel', 'JobInvolvementLabel', 'JobSatisfactionLabel',
             'RelationshipSatisfactionLabel', 'WorkLifeBalanceLabel', 'PerformanceRatingLabel',
             'AgeGroup', 'TenureGroup']

X = df_clean.drop(columns=drop_cols)
y = df_clean['AttritionFlag']

# Encode categorical columns
cat_cols = X.select_dtypes(include='object').columns
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

X.shape, y.shape

C:\Users\timma\AppData\Local\Temp\ipykernel_1988\2606600694.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include='object').columns


((1470, 30), (1470,))

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

((1176, 30), (294, 30))

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.92      0.66      0.77       247
           1       0.28      0.70      0.40        47

    accuracy                           0.67       294
   macro avg       0.60      0.68      0.59       294
weighted avg       0.82      0.67      0.71       294

ROC-AUC: 0.7271082780601258


C:\Users\timma\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.94      0.75      0.84       247
           1       0.37      0.77      0.50        47

    accuracy                           0.75       294
   macro avg       0.66      0.76      0.67       294
weighted avg       0.85      0.75      0.78       294

ROC-AUC: 0.8073908174692048


In [16]:
os.makedirs('models', exist_ok=True)

In [17]:
import joblib

joblib.dump(model, 'models/attrition_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
joblib.dump(label_encoders, 'models/label_encoders.pkl')
joblib.dump(list(X.columns), 'models/feature_columns.pkl')

['models/feature_columns.pkl']